In [ ]:
dbutils.widgets.text("catalog_param", "my_assessment")
dbutils.widgets.text("schema_param", "gold")

catalog = dbutils.widgets.get("catalog_param")
schema = dbutils.widgets.get("schema_param")

In [ ]:
from pyspark.sql.functions import sum, count, col, month, year, broadcast

In [ ]:
customers_df   = spark.table(f"{catalog}.silver.customers_cleaned")
products_df    = spark.table(f"{catalog}.silver.products_cleaned")
orders_df      = spark.table(f"{catalog}.silver.orders_cleaned")
order_items_df = spark.table(f"{catalog}.silver.order_items_cleaned")

In [ ]:
customers_df.cache()
products_df.cache()

In [ ]:
# Broadcast small tables (products, customers) to avoid shuffle
fact_sales = order_items_df \
    .join(broadcast(products_df), "product_id") \
    .join(orders_df, "order_id") \
    .join(broadcast(customers_df), "customer_id") \
    .select(
        col("order_item_id"),
        col("order_id"),
        col("customer_id"),
        col("product_id"),
        col("product_name"),
        col("category"),
        col("quantity"),
        col("order_items_df.price").alias("unit_price"),
        col("total_amount"),
        col("order_date"),
        col("order_status"),
        col("city"),
        col("state")
    )

fact_sales.write.mode("overwrite") \
    .partitionBy("order_date") \
    .saveAsTable(f"{catalog}.gold.fact_sales")

print("fact_sales created")

In [ ]:
dim_customers = customers_df.select(
    "customer_id", "name", "city", 
    "state", "signup_date"
)

dim_customers.write.mode("overwrite") \
    .saveAsTable(f"{catalog}.gold.dim_customers")

print("dim_customers created")

In [ ]:
dim_products = products_df.select(
    "product_id", "product_name", 
    "category", "price"
)

dim_products.write.mode("overwrite") \
    .saveAsTable(f"{catalog}.gold.dim_products")

print("dim_products created")

In [ ]:
revenue_by_state = fact_sales \
    .groupBy("state") \
    .agg(sum("total_amount").alias("total_revenue"),
         count("order_id").alias("total_orders")) \
    .orderBy(col("total_revenue").desc())

revenue_by_state.write.mode("overwrite") \
    .saveAsTable(f"{catalog}.gold.revenue_by_state")

print("✅ revenue_by_state created")

In [ ]:
top_products = fact_sales \
    .groupBy("product_id", "product_name", "category") \
    .agg(sum("total_amount").alias("total_revenue"),
         sum("quantity").alias("total_quantity")) \
    .orderBy(col("total_revenue").desc())

top_products.write.mode("overwrite") \
    .saveAsTable(f"{catalog}.gold.top_products")

print("✅ top_products created")

In [ ]:
sales_trends = fact_sales \
    .withColumn("month", month("order_date")) \
    .withColumn("year", year("order_date")) \
    .groupBy("year", "month", "order_date") \
    .agg(sum("total_amount").alias("daily_revenue"),
         count("order_id").alias("daily_orders")) \
    .orderBy("year", "month", "order_date")

sales_trends.write.mode("overwrite") \
    .saveAsTable(f"{catalog}.gold.sales_trends")

print("✅ sales_trends created")